<a href="https://colab.research.google.com/github/iam4tart/speech-lab/blob/main/03-true-streaming-causal-model/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, random, math, string
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import pandas as pd
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
torch.backends.cudnn.benchmark = True

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

In [ ]:
CFG = {
    "sample_rate": 16000, "batch_size": 16, "epochs": 30, "lr": 5e-4,
    "embedding_dim": 32, "num_transformer_layers": 6, "num_heads": 4,
    "strides": (2, 2, 2, 2),
}

In [ ]:
os.makedirs("./data", exist_ok=True)
!wget -q https://www.openslr.org/resources/104/Hindi-English_train.tar.gz
!tar -xzf Hindi-English_train.tar.gz -C ./data/
print(os.listdir("./data"))

In [ ]:
devanagari = [chr(c) for c in range(0x0900, 0x097F)]
characters = devanagari + list(string.ascii_uppercase) + [" "]
vocab = characters + ["<blank>"]
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
blank_idx = char_to_idx["<blank>"]
vocab_size = len(vocab)
print("Vocab size:", vocab_size)

In [ ]:
class STTDataset(Dataset):
    def __init__(self, audio_paths, token_sequences):
        self.audio_paths, self.token_sequences = audio_paths, token_sequences
    def __len__(self): return len(self.audio_paths)
    def __getitem__(self, idx):
        waveform, sr = torchaudio.load(self.audio_paths[idx])
        if waveform.shape[0] > 1: waveform = waveform.mean(dim=0, keepdim=True)
        if sr != 16000: waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
        waveform = waveform.squeeze(0)
        tokens = torch.tensor(self.token_sequences[idx], dtype=torch.long)
        return waveform, tokens

def collate_fn(batch):
    waveforms, tokens = zip(*batch)
    audio_lengths = torch.tensor([w.shape[0] for w in waveforms], dtype=torch.long)
    target_lengths = torch.tensor([len(t) for t in tokens], dtype=torch.long)
    waveforms = torch.nn.utils.rnn.pad_sequence(waveforms, batch_first=True)
    tokens = torch.nn.utils.rnn.pad_sequence(tokens, batch_first=True)
    return waveforms, tokens, audio_lengths, target_lengths

In [ ]:
class ResidualDownSampleBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride, kernel_size=8):
        super().__init__()
        padding_val = (kernel_size - 1) // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding_val)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, stride=stride, padding=padding_val)
        self.relu = nn.ReLU()
        if in_channels != out_channels or stride != 1:
            self.residual_proj = nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.residual_proj = nn.Identity()
    def forward(self, x):
        residual = self.residual_proj(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.conv2(out)
        if out.shape[-1] != residual.shape[-1]:
            min_len = min(out.shape[-1], residual.shape[-1])
            out, residual = out[..., :min_len], residual[..., :min_len]
        return self.relu(out + residual)

class DownsamplingNetwork(nn.Module):
    def __init__(self, embedding_dim=32, hidden_dim=16, in_channels=1, strides=(2,2,2,2)):
        super().__init__()
        self.mean_pooling = nn.AvgPool1d(kernel_size=2, stride=2)
        self.layers = nn.ModuleList()
        current_in = in_channels
        for i, s in enumerate(strides):
            block_in = current_in if i == 0 else hidden_dim
            self.layers.append(ResidualDownSampleBlock(block_in, hidden_dim, stride=s))
        self.final_conv = nn.Conv1d(hidden_dim, embedding_dim, kernel_size=4, padding="same")
    def forward(self, x):
        x = self.mean_pooling(x)
        for layer in self.layers: x = layer(x)
        x = self.final_conv(x)
        return x.transpose(1, 2)

In [ ]:
class SinusoidalPositionEncoding(nn.Module):
    def __init__(self, embed_size, max_seq_length=10000):
        super().__init__()
        position = torch.arange(max_seq_length).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_size, 2) * (-math.log(10000.0) / embed_size))
        pe = torch.zeros(max_seq_length, embed_size)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("positional_embedding", pe)
    def forward(self, x): return x + self.positional_embedding[:x.size(1), :]

class FeedForward(nn.Module):
    def __init__(self, embed_size, ff_hidden_mult=4, dropout=0.1):
        super().__init__()
        hidden = ff_hidden_mult * embed_size
        self.layer1 = nn.Linear(embed_size, hidden)
        self.layer2 = nn.Linear(hidden, embed_size)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x): return self.layer2(self.dropout(F.gelu(self.layer1(x))))

In [ ]:
class CausalSelfAttentionLayer(nn.Module):
    def __init__(self, embed_size, num_heads, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_size, num_heads, dropout=dropout, batch_first=True)
        self.attn_dropout = nn.Dropout(dropout)
        self.attn_norm = nn.LayerNorm(embed_size)
        self.ff = FeedForward(embed_size, dropout=dropout)
        self.ff_dropout = nn.Dropout(dropout)
        self.ff_norm = nn.LayerNorm(embed_size)
    def forward(self, x):
        T = x.shape[1]
        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        attn_out, _ = self.mha(x, x, x, attn_mask=causal_mask, need_weights=False)
        x = self.attn_norm(x + self.attn_dropout(attn_out))
        ff_out = self.ff(x)
        return self.ff_norm(x + self.ff_dropout(ff_out))

class CausalTransformerEncoder(nn.Module):
    def __init__(self, embed_size=32, num_layers=6, num_heads=4, max_seq_length=10000):
        super().__init__()
        self.positional_encoding = SinusoidalPositionEncoding(embed_size, max_seq_length)
        self.blocks = nn.ModuleList([CausalSelfAttentionLayer(embed_size, num_heads) for _ in range(num_layers)])
        self.dropout = nn.Dropout(0.1)
    def forward(self, x):
        x = self.dropout(self.positional_encoding(x))
        for block in self.blocks: x = block(x)
        return x

In [ ]:
class CausalTranscribeModel(nn.Module):
    def __init__(self, embedding_dim=32, vocab_size=28, strides=(2,2,2,2),
                 num_transformer_layers=6, num_heads=4, max_seq_length=10000):
        super().__init__()
        self.downsampling_network = DownsamplingNetwork(
            embedding_dim=embedding_dim, hidden_dim=embedding_dim // 2,
            in_channels=1, strides=strides
        )
        self.encoder = CausalTransformerEncoder(
            embed_size=embedding_dim, num_layers=num_transformer_layers,
            num_heads=num_heads, max_seq_length=max_seq_length
        )
        self.output_layer = nn.Linear(embedding_dim, vocab_size)
    def forward(self, x):
        if x.dim() == 2: x = x.unsqueeze(1)
        x = self.downsampling_network(x)
        x = self.encoder(x)
        x = self.output_layer(x)
        return F.log_softmax(x, dim=-1)

In [ ]:
model = CausalTranscribeModel(
    embedding_dim=CFG["embedding_dim"], vocab_size=vocab_size,
    strides=CFG["strides"], num_transformer_layers=CFG["num_transformer_layers"],
    num_heads=CFG["num_heads"],
).to(device)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

In [ ]:
scaler = torch.amp.GradScaler("cuda")
optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
ctc_loss_fn = torch.nn.CTCLoss(blank=blank_idx, zero_infinity=True)
num_epochs = CFG["epochs"]
best_loss = float("inf")
os.makedirs("checkpoints", exist_ok=True)

start_epoch = 0
if os.path.exists("checkpoints/last.pth"):
    ckpt = torch.load("checkpoints/last.pth", map_location=device)
    (model.module if isinstance(model, nn.DataParallel) else model).load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    print(f"Resumed from epoch {start_epoch}")

for epoch in range(start_epoch, num_epochs):
    model.train()
    total_loss = 0.0
    for i, (waveforms, tokens, audio_lengths, target_lengths) in enumerate(loader):
        waveforms, tokens, target_lengths = waveforms.to(device), tokens.to(device), target_lengths.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            log_probs = model(waveforms)
            input_lengths = torch.full((log_probs.shape[0],), log_probs.shape[1], dtype=torch.long, device=device)
            loss = ctc_loss_fn(log_probs.permute(1,0,2), tokens, input_lengths, target_lengths)
        if torch.isnan(loss) or torch.isinf(loss): continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
        if i % 100 == 0: print(f"Epoch [{epoch+1}/{num_epochs}] Step [{i}] Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Avg Loss: {avg_loss:.4f}")
    model_to_save = model.module if isinstance(model, nn.DataParallel) else model
    torch.save({"epoch": epoch, "model_state_dict": model_to_save.state_dict(),
                "optimizer_state_dict": optimizer.state_dict()}, "checkpoints/last.pth")
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model_to_save.state_dict(), "checkpoints/best_model.pth")
        print("best model saved")